# Cody, Kumar & Tzounopoulos (2024) on dandiset 001256

**Paper.** Cody P, Kumar M, Tzounopoulos T (2024). Cortical zinc signaling is necessary for changes in
mouse pupil diameter that are evoked by background sounds with different contrasts. J Neurosci 44(11)
e0939232024, doi:10.1523/JNEUROSCI.0939-23.2024.

**Dataset.** DANDI:001256, version 0.241120.2150 (doi:10.48324/dandi.001256/0.241120.2150), streamed from
the archive. The paper cites no version (it names no dandiset); 0.241120.2150 is the only published
version and the most recent, and the draft has the same 30 assets and size, so it is pinned.

**Claims.** `claims/001256/10.1523-jneurosci.0939-23.2024.yaml`, written from the version of record
(main text, Methods and figure legends; the paper has no supplement and links no code).

| Claim | Tested |
|---|---|
| fig1f-tone-response-inverse-u | tested |
| fig1g-reliability-intermediate | tested |
| fig1h-cgc-largest-intermediate | tested |
| fig1h-cgc-evident-all-bins | tested |
| fig1i-baseline-contrast-interaction | tested |
| fig1i-contrast-effect-intermediate-only | tested |
| fig2b-evoked-pd-whole-drc | tested |
| fig2c-evoked-pd-1-2s | tested; blocked (readings disagree, as the criterion says) |
| fig2e-drc-responses-no-difference | tested |
| fig3b-lme-coefficients | tested |
| fig3c-lme-likelihood-ratio | tested |
| fig4c-wt-contrast-zx1-effects | tested |
| fig4-wt-posthoc | tested |
| fig4f-ko-no-contrast-effect | tested; blocked (readings disagree, as the criterion says) |
| fig4f-ko-zx1-slope | tested |
| fig5-contralateral-zx1-increases | blocked, data missing (no widefield imaging in the dandiset) |
| fig5-ipsilateral-zx1-reduces | blocked, data missing |
| fig5c-no-dye-in-other-hemisphere | blocked, data missing |
| fig6a-baseline-pd-drift | tested |
| fig6b-acsf-contrast-effect | tested |
| fig6c-no-baseline-evoked-correlation | tested; blocked (no listed reading matches the count, and they disagree) |
| methods-baseline-evoked-correlation | tested; blocked (no reading matches the counts, and they disagree) |
| fig6d-ko-reduced-high-contrast | tested |
| discussion-zx1-wt-like-ko | tested |

Where a method detail is not in the claims file it is taken from the paper's Methods, cited in the helper
module `zinc_pupil.py`. No random numbers are used. Loading takes about one to two minutes.

Revised after the first verification round: the pupil cleaning also drops trials with tracking-failure
steps (Data check), the growth curve fits use several optimizers and print their diagnostics, and variants
for the cleaning and for group A without its ZnT3-WT mouse are at the end. Revised after the second
round: a variant with the step rule counted after the spike filter, the Fig. 6a overlap under every ZX1
reading and cleaning, the Fig. 4c criterion per reading and cleaning, and two proposed changes of
outcome (fig4c-wt-contrast-zx1-effects and fig4-wt-posthoc). The third verification round agreed with both,
and the two outcomes were changed after it.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

import zinc_pupil as zp

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
T0 = time.time()
print("dandiset", zp.DANDISET_ID, "version", zp.DANDISET_VERSION)

In [ ]:
assets = zp.list_assets()
meta, trials = zp.load_all(assets)
print(f"{len(assets)} files, {len(trials)} trials read in {time.time() - T0:.0f} s")

## Data check

What was loaded, by subject, with the analysis labels the files carry in `session_description`.
The groups of the claims file are taken from these labels (A: `pupilPTinContrast`; C:
`pupil_contrast_ZnT3WT`; D: `pupil_contrast_ZnT3KO`; F: `pupil_contrast_ACSF`), and group B (Fig. 2d, e,
cells before and after ZX1) as the mice with CGC-assay trials both before and after ZX1, which no label
names.

In [ ]:
meta["labels"] = meta.session_description.str.split(" | ")
lab = meta.set_index("subject").labels
def in_label(name):
    return sorted(s for s, l in lab.items() if name in l)

GROUPS = {
    "A": in_label("pupilPTinContrast"),
    "C": in_label("pupil_contrast_ZnT3WT"),
    "D": in_label("pupil_contrast_ZnT3KO"),
    "F": in_label("pupil_contrast_ACSF"),
    "ZX1_drift": in_label("ZX1_uBasePupilPctMax_wEpoch"),
    "ACSF_drift": in_label("ACSF_uBasePupilPctMax_wEpoch"),
    "base_vs_evoked": in_label("pupilBaseline_vs_evoked"),
}
trials["treat"] = trials.treatment.map({"none": "CTRL", "preZX1": "CTRL", "preACSF": "CTRL",
                                        "postZX1": "ZX1", "postACSF": "ACSF"})
pt_tab = pd.crosstab(trials[trials.kind == "pt"].subject, trials[trials.kind == "pt"].treat)
GROUPS["B"] = sorted(pt_tab.index[(pt_tab.get("CTRL", 0) > 0) & (pt_tab.get("ZX1", 0) > 0)])
summary = meta[["subject", "genotype", "sex", "age"]].set_index("subject")
for g, subs in GROUPS.items():
    summary[g] = summary.index.isin(subs).astype(int)
print(summary.sort_index().to_string())
for g, subs in GROUPS.items():
    sx = summary.loc[subs, "sex"].value_counts().to_dict()
    gt = summary.loc[subs, "genotype"].value_counts().to_dict()
    print(f"group {g}: {len(subs)} mice, sex {sx}, genotype {gt}")
print()
print("trials by kind and treatment")
print(pd.crosstab(trials.kind, trials.treat))
print()
print("trials by contrast of the first DRC block (kind pt and drc)")
print(pd.crosstab(trials[trials.kind.isin(["pt", "drc"])].kind, trials.contrast))
print("stimDelay values (s):", trials.groupby("kind").stimDelay.unique().to_dict())
print("2P frame rates:", trials.frameRate.unique(), "frames per trial:", trials.groupby("kind").nFrames.unique().to_dict())
st = trials.starting_time.astype(float)
ft = pd.to_datetime(trials.fileTimeInstantiate)
off = (ft - ft.groupby(trials.subject).transform(lambda x: x[st.loc[x.index] == 0].min())).dt.total_seconds()
print("trials whose table starting_time differs from their .tif creation time (relative to the reference .tif) by more than 1 s:",
      int(((st - off).abs() > 1).sum()), "; by subject:", trials[(st - off).abs() > 1].subject.value_counts().to_dict())
print("  of these, starting_time - (creation time offset) in hours:", sorted(((st - off)[(st - off).abs() > 1] / 3600).round(2).unique().tolist())[:5])
print("pure-tone frequencies of the CGC assay:", trials[trials.kind == "pt"].pulseSets.str.extract(r"(\d+Hz)")[0].value_counts().to_dict())

Checks of the metadata against the values:

- Pupil series: two video frames per two-photon frame (10 and 5 Hz) is expected. Pupil series whose
  length does not match their trial (a series of a mapping trial's length attached to a DRC trial)
  are counted and left out. Every pupil series's description names `TwoPhotonSeries_036` (or the
  file's last series); the series is matched to its trial by the .tif file named in its `comments`.
- Units: the pupil radius is in "na" (pixels, judged from the values) and the fluorescence in
  "lumens" (raw intensity); both analyses use ratios, so the units do not enter.
- Raw pupil traces hold dropped frames (NaN) and tracking failures.

In [ ]:
snd = trials[trials.kind.isin(["pt", "drc"])].copy()
snd["has_pupil"] = snd.pupil.notna()
snd["len_ok"] = [p is not None and len(p) == 2 * int(n) for p, n in zip(snd.pupil, snd.nFrames)]
print("sound trials:", len(snd), "with pupil:", snd.has_pupil.sum(), "pupil length = 2 x frames:", snd.len_ok.sum())
print("mismatched pupil lengths by subject:", snd[snd.has_pupil & ~snd.len_ok].subject.value_counts().to_dict())
raw = snd[snd.len_ok]
nan_frac = raw.pupil.map(lambda p: np.isnan(p).mean())
print(f"fraction of pupil frames missing per trial: median {nan_frac.median():.3f}, "
      f"trials all missing {int((nan_frac == 1).sum())}")
rmax = raw.pupil.map(lambda p: np.nanmax(p) if np.isfinite(p).any() else np.nan)
rmed = raw.pupil.map(lambda p: np.nanmedian(p) if np.isfinite(p).any() else np.nan)
print(f"raw radius: median of trial medians {rmed.median():.1f}, range of trial medians "
      f"{rmed.min():.1f} to {rmed.max():.1f}; trials whose maximum exceeds 3 x their median: "
      f"{int((rmax > 3 * rmed).sum())} (largest raw value {np.nanmax(rmax):.3g})")

# Pupil cleaning: the paper's single pass, and the pass repeated with a spike filter (main).
single = zp.pupil_measures(trials, jump_ratio=None, max_pass=1, spike_filter=False, max_missing=1.0)
prev = zp.pupil_measures(trials, jump_ratio=None)  # round-1 cleaning: no trial dropped for jumps
tr = zp.pupil_measures(trials)
# narrower form of the step rule (variant): steps counted after the spike filter
narrow = zp.pupil_measures(trials, jump_after_filter=True)
print(f"trials with a usable pupil: single pass {len(single)}, round-1 cleaning {len(prev)}, main cleaning {len(tr)}")
print("trials dropped for a step by more than a factor 2 between consecutive tracked frames:",
      int((prev.n_jumps > 0).sum()), "; by subject:", prev[prev.n_jumps > 0].subject.value_counts().to_dict())
print(f"trials dropped for a step, {int((prev.n_jumps > 0).sum())} in "
      f"{prev[prev.n_jumps > 0].subject.nunique()} mice; counting steps after the spike filter (variant): "
      f"{len(prev) - len(narrow)} in {prev[~prev.trial_id.isin(narrow.trial_id)].subject.nunique()} mice, by subject:",
      prev[~prev.trial_id.isin(narrow.trial_id)].subject.value_counts().to_dict())
amax = pd.DataFrame({
    "max_single_pass": single.groupby("subject").pclean.apply(lambda s: max(np.max(x) for x in s)),
    "max_round1": prev.groupby("subject").pclean.apply(lambda s: max(np.max(x) for x in s)),
    "max_main": tr.groupby("subject").pclean.apply(lambda s: max(np.max(x) for x in s)),
    "max_narrow": narrow.groupby("subject").pclean.apply(lambda s: max(np.max(x) for x in s)),
    "median_main": tr.groupby("subject").pclean.apply(lambda s: np.median(np.concatenate(list(s)))),
    "trials_main": tr.groupby("subject").size(),
})
print(amax.round(1).to_string())
# residual artifacts: the five largest cleaned trial maxima per animal, each with its trial's median
tm = tr.assign(tmax=tr.pclean.map(np.max), tmed=tr.pclean.map(np.median)).sort_values("tmax", ascending=False)
top5 = tm.groupby("subject").head(5).groupby("subject").apply(
    lambda g: ", ".join(f"{a:.0f}/{b:.0f}" for a, b in zip(g.tmax, g.tmed)))
print("five largest trial maxima / that trial's median, main cleaning:")
print(top5.to_string())
print("largest ratio of a trial's maximum to the animal's median:",
      (tm.tmax / tm.subject.map(amax.median_main)).max().round(2))
print("baseline PD (% animal max), main cleaning: range", tr.base_pct_max.min().round(1), "to",
      tr.base_pct_max.max().round(1), "; below 50%:", int((tr.base_pct_max < 50).sum()), "of", len(tr))

With one pass, several animals keep pupil "maxima" of 200 to 550000 pixels, against medians of 20 to 100,
so a baseline in % of the animal's maximum would be near 0 for every trial of those animals. The round-1
cleaning (Hampel spike filter, the derivative step repeated, and trials with more than half of their frames
missing dropped) still left tracking failures in two mice: in AA0319 and AA0334 the failures come as runs
at a second level (130 to 160, against a pupil of about 25), which set their maxima at 332.7 and 162.1. The
main cleaning also drops every trial whose raw trace steps by more than a factor 2 between consecutive
tracked frames (118 trials, 15 mice); a pupil does not double or halve its radius in 100 ms. This changes
the maximum of those two mice only (78.3 and 70.9), and the largest trial maximum is now at most 3.1 times
its animal's median. The rule reads the raw trace, before the spike filter, so it also drops trials whose
only fault is a single-frame spike that the filter removes: it is broader than the two mice need. Counting
the steps after the spike filter drops 39 trials in 6 mice and gives the same animal maxima (`max_narrow`);
it is run as a variant at the end. Where an outcome differs between the two, the difference comes from
the 79 trials in other mice that only the broad rule drops. All three steps are assumptions; the paper's own DeepLabCut points and likelihoods,
which would let the frames be dropped as it describes, are not in the files. The other cleanings are run
as variants at the end of the notebook.

Clock check. The pupil series start at their trial's start time (asserted on loading). Sound onset is
`stimDelay` after the first frame. If this is right, the population calcium response should rise at the
pure tone (2 s after onset, in both contrasts) and not at other times, and the pupil should dilate
after onset, not before. Shifting the assumed onset by whole seconds moves the tone response away from
2 s. The DRC structure (sound-onset delays, which contrast comes first, mapping trials) is printed after
the duplicate check.


In [ ]:
ptr = trials[trials.kind == "pt"]
pop = []
for _, r in ptr.iterrows():
    d, t = zp.dff(np.asarray(r.F, float), r.stimDelay)
    pop.append(np.nanmean(d, 1))
pop = np.nanmean(np.vstack(pop), 0)
t = zp.frame_times(len(pop), 4)
print("population mean dF/F (%) of CGC-assay trials by time from sound onset:")
print(pd.Series(pop.round(2), index=t.round(1)).loc[-1.2:4.0].to_string())
d1 = np.diff(pop)
k = np.argmax(d1[(t[:-1] >= 0.5)]) + np.sum(t[:-1] < 0.5)
print(f"largest frame-to-frame rise after 0.5 s: from {t[k]:.1f} to {t[k+1]:.1f} s (tone at 2.0 s)")
ev = tr[tr.kind.isin(["pt", "drc"])]
m = np.nanmean(np.vstack([x[:200] for x in ev.ev_trace]), 0)
tp = np.arange(200) / 10 - 4
print("mean pupil (% baseline) over trials: -4 to 0 s", m[(tp >= -4) & (tp < 0)].mean().round(2),
      "; 0 to 1 s", m[(tp >= 0) & (tp < 1)].mean().round(2), "; 1 to 2 s", m[(tp >= 1) & (tp < 2)].mean().round(2),
      "; 2 to 4 s", m[(tp >= 2) & (tp < 4)].mean().round(2))
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(t, pop); ax[0].axvline(0, c="k", ls=":"); ax[0].axvline(2, c="g", ls=":")
ax[0].set(xlabel="time from sound onset (s)", ylabel="mean dF/F (%)", title="CGC assay, all cells")
ax[1].plot(tp, m); ax[1].axvline(0, c="k", ls=":")
ax[1].set(xlabel="time from sound onset (s)", ylabel="pupil (% baseline)", title="all sound trials")
plt.tight_layout(); plt.show()
print("plotted ranges: dF/F", pop.min().round(2), "to", pop.max().round(2), "; pupil", m.min().round(2), "to", m.max().round(2))

In [ ]:
# Recordings deposited twice: identical pupil traces or fluorescence across trials.
import hashlib
def h(a):
    a = np.asarray(a, float)
    return None if not np.isfinite(a).any() else hashlib.md5(np.nan_to_num(a).tobytes()).hexdigest()
hp = raw.pupil.map(h).dropna()
hf = ptr.F.map(h)
print("identical pupil traces (excluding all-missing):", int(hp.duplicated().sum()),
      "; identical fluorescence arrays:", int(hf.duplicated().sum()))
print("subjects:", meta.subject.nunique(), "files:", len(meta))

In [ ]:
# DRC trial structure: sound-onset delay, first block, and mapping trials, by subject
snd_all = trials[trials.kind.isin(["pt", "drc"])]
first_block = snd_all.pulseNames.str.extract(r"(\d\d-\d\ddB_to_\d\d-\d\ddB)")[0].fillna("one contrast")
print("trials by kind, stimDelay and first-block order (all subjects):")
print(pd.crosstab([snd_all.kind, snd_all.stimDelay], first_block).to_string())
drc_all = trials[trials.kind == "drc"]
print("DRC trials (no tone) by subject, treatment and contrast of the first block:")
print(pd.crosstab(drc_all.subject, [drc_all.treat, drc_all.contrast]).to_string())
print("DRC trials with stimDelay = 10 s by subject and treatment:",
      drc_all[drc_all.stimDelay == 10].groupby(["subject", "treat"]).size().to_dict())
print("FRA mapping trials by subject:", trials[trials.kind == "map"].subject.value_counts().to_dict())
print("  of these, group A mice:", sorted(set(trials[trials.kind == "map"].subject) & set(GROUPS["A"])))
# Clock check for the stimDelay = 10 s trials: the pupil should dilate after the onset taken from stimDelay
# (10 s), not after 4 s, the onset of the other DRC trials.
d10 = tr[(tr.kind == "drc") & (tr.stimDelay == 10)]
for onset in (10, 4):
    m10 = np.nanmean(np.vstack(list(d10.pclean)), 0)
    tt = np.arange(len(m10)) / 10 - onset
    b = m10[(tt >= -4) & (tt < 0)].mean()
    print(f"stimDelay-10 DRC trials ({len(d10)}), onset taken at {onset} s: mean pupil (% of -4 to 0 s) over 0 to 2 s "
          f"{100 * m10[(tt >= 0) & (tt < 2)].mean() / b:.2f}, over 2 to 4 s {100 * m10[(tt >= 2) & (tt < 4)].mean() / b:.2f}")


## Figure 1: pure-tone responses, reliability and contrast scaling across baseline PD (group A)

Group A is the ten mice the files label `pupilPTinContrast` (the paper gives eight, four F and four M; the
files give ten, four F and six M, one of them ZnT3-WT). Keeping the ZnT3-WT mouse AA0335, as the label
does, is an assumption; group A without it is a variant at the end, and the Fig. 2b and Fig. 3b outcomes
rest on this choice. CTRL trials of the CGC assay (DRC with the pure
tone) are used. Per trial: the pure-tone response trace (dF/F minus its 1-2 s mean), the peak in each
window, and the trial's baseline PD in % of the animal's maximum.

Readings run (claims file, shared methods): cell inclusion, SD over time of the trial-averaged trace
(`time`) or across trials (`trials`), and responsive in both contrasts (`both`) or with contrasts
pooled (`pooled`); peak window 2-2.8 s or 2-3.2 s; peak as the maximum of the cell's trial-averaged
trace in the bin (`avgmax`) or the mean of its trial maxima (`meanmax`); bin 1 all trials below 21%
(`open`) or 10 to 21% (`10-21`). The FRA-mapping reading of inclusion (claims file, (b)) cannot be run:
mapping trials are in 7 of 30 files and only one group A mouse (AA0318; counts by subject in the Data check).

In [ ]:
A = GROUPS["A"]
pt_ctrl = trials[(trials.kind == "pt") & (trials.treat == "CTRL") & trials.subject.isin(A)]
ct = zp.cell_trial_table(pt_ctrl)
base = tr.set_index("trial_id").base_pct_max
ct["base"] = ct.trial.map(base)
print("group A CTRL CGC-assay trials:", pt_ctrl.trial_id.nunique(), "; with a usable pupil:",
      int(pt_ctrl.trial_id.isin(base.index).sum()))
print("cells (ROIs) in group A:", ct.cell.nunique(), "; by mouse:", ct.groupby("subject").cell.nunique().to_dict())
INCL = {}
for sd in ("time", "trials"):
    for both in (True, False):
        INCL[(sd, "both" if both else "pooled")] = zp.responsive(ct, (2.0, 2.8), sd=sd, by_contrast=both)
print("tone-responsive cells under each inclusion reading (paper: 295 from eight mice):")
for k, v in INCL.items():
    print(f"  SD {k[0]:6s} {k[1]:6s}: {len(v)} cells")
ctb = ct[ct.base.notna()].copy()
print("cell-trials without a usable pupil (dropped):", int(ct.base.isna().sum()), "of", len(ct))

def bin_values(d, peak_col, window, mode, lower, by=("cell", "bin")):
    """Average peak per cell and bin (and contrast when in `by`)."""
    d = d.copy()
    d["bin"] = zp.pd_bin(d.base.values, 10 if lower == "10-21" else None)
    d = d[d.bin >= 0]
    out = []
    for key, g in d.groupby(list(by)):
        if mode == "meanmax":
            v = g[peak_col].mean()
        else:
            tr_ = np.vstack(g.trace.values).mean(0)
            t_ = g.t.iloc[0]
            v = tr_[(t_ >= window[0]) & (t_ < window[1])].max()
        out.append(dict(zip(by, key if isinstance(key, tuple) else (key,)), value=v, n=len(g)))
    return pd.DataFrame(out)

def kw(df):
    groups = [g.value.values for _, g in df.groupby("bin")]
    H, p = stats.kruskal(*groups)
    means = df.groupby("bin").value.mean()
    return H, p, int(means.idxmax()), means

WINDOWS = {"2-2.8": (2.0, 2.8), "2-3.2": (2.0, 3.2)}
rows = []
for (sd, bc), cells in INCL.items():
    d0 = ctb[ctb.cell.isin(cells)]
    for wn, w in WINDOWS.items():
        for mode in ("avgmax", "meanmax"):
            for lower in ("open", "10-21"):
                v = bin_values(d0, f"peak_{w[0]}_{w[1]}", w, mode, lower)
                H, p, top, means = kw(v)
                rows.append({"SD": sd, "contrasts": bc, "window": wn, "peak": mode, "bin1": lower,
                             "cells": v.cell.nunique(), "cell_bins": len(v), "H": round(H, 2), "p": p,
                             "top_bin": zp.BIN_LABELS[top], "inverse_U": 0 < top < 6,
                             **{f"m{b}": round(means.get(b, np.nan), 2) for b in range(7)}})
F1F = pd.DataFrame(rows)
print(F1F.to_string())
print("readings with p < 0.05 and an interior top bin:", int(((F1F.p < 0.05) & F1F.inverse_U).sum()), "of", len(F1F))

In [ ]:
# Main reading (first listed): SD over time, both contrasts, 2-2.8 s, avgmax, bin 1 open; by mouse.
main_cells = INCL[("time", "both")]
v = bin_values(ctb[ctb.cell.isin(main_cells)], "peak_2.0_2.8", (2.0, 2.8), "avgmax", "open")
v["subject"] = v.cell.str.split("/").str[0]
print("cell-bins per bin:", v.groupby("bin").size().to_dict())
print("mean peak (% dF/F) by mouse and bin:")
print(v.pivot_table(index="subject", columns="bin", values="value", aggfunc="mean").round(1).to_string())
print("cells and bins per mouse:", v.groupby("subject").cell.nunique().to_dict())
fig, ax = plt.subplots(figsize=(4, 3))
mm = v.groupby("bin").value.agg(["mean", "sem"])
ax.errorbar([zp.BIN_LABELS[i] for i in mm.index], mm["mean"], mm["sem"], marker="o")
ax.set(xlabel="baseline PD (% animal max)", ylabel="peak pure-tone dF/F (%)"); plt.xticks(rotation=45); plt.tight_layout(); plt.show()
print("plotted means range", mm["mean"].min().round(2), "to", mm["mean"].max().round(2))

In [ ]:
print("trials with baseline PD below 10% of animal max:", int((ctb.drop_duplicates("trial").base < 10).sum()),
      "; so the two bin-1 readings are the same here")

**Result: reproduced.** Under the main reading (SD over time, both contrasts, 2-2.8 s, maximum of the averaged trace) 318 cells are tone-responsive, within the tolerance of 295 (265.5 to 324.5); the other inclusion readings give 339, 215 and 241. Peak responses differ across the seven baseline PD bins (Kruskal-Wallis H = 51.83, p = 2.0e-09), with the largest mean in the 21-32% bin (15.88% dF/F) against 8.12 in the lowest bin and 13.08 in the highest. All 32 readings give p < 0.05 (1.0e-10 to 7.3e-08) and the largest mean in the 21-32% bin. The bins are not independent of mouse: the lowest bin holds cells of three mice only (AA0335, AA0350, AA0353) and the highest of five. Under the other cleanings, and without the ZnT3-WT mouse AA0335, the main reading gives p = 2.8e-10 to 3.2e-06 with an interior top bin (variants at the end).

### Reliability (Fig. 1g)

Per cell and bin, the mean over all pairs of the cell's trials in the bin of the zero-lag correlation of
the pure-tone response traces at 2 to 3.2 s (six frames at 5 Hz): (a) Pearson, (b) normalized without
mean removal.

In [ ]:
def reliability(d, how):
    d = d.copy()
    d["bin"] = zp.pd_bin(d.base.values)
    out = []
    for (cell, b), g in d.groupby(["cell", "bin"]):
        if len(g) < 2:
            continue
        t_ = g.t.iloc[0]
        X = np.vstack(g.trace.values)[:, (t_ >= 2.0) & (t_ < 3.2)]
        if how == "pearson":
            X = X - X.mean(1, keepdims=True)
        nrm = np.linalg.norm(X, axis=1)
        ok = nrm > 0
        X = X[ok] / nrm[ok, None]
        if len(X) < 2:
            continue
        C = X @ X.T
        iu = np.triu_indices(len(X), 1)
        out.append({"cell": cell, "bin": b, "value": C[iu].mean(), "n": len(X)})
    return pd.DataFrame(out)

rows = []
for (sd, bc), cells in INCL.items():
    for how in ("pearson", "no_mean_removal"):
        v = reliability(ctb[ctb.cell.isin(cells)], how)
        H, p, top, means = kw(v)
        rows.append({"SD": sd, "contrasts": bc, "cc": how, "cells": v.cell.nunique(), "cell_bins": len(v),
                     "H": round(H, 2), "p": p, "top_bin": zp.BIN_LABELS[top], "interior": 0 < top < 6,
                     **{f"m{b}": round(means.get(b, np.nan), 3) for b in range(7)}})
F1G = pd.DataFrame(rows)
print(F1G.to_string())
print("cell-bins with fewer than two trials have no value")

**Result: reproduced.** Reliability differs across bins under all eight readings (Kruskal-Wallis p = 1.1e-09 to 6.2e-07), and the bin with the highest mean is 21-32% in every reading (Pearson, main inclusion: 0.766 against 0.633 in the lowest bin and 0.590 in the highest).

### Contrast scaling factor (Fig. 1h)

Per cell (responsive in both contrasts) and bin with trials of both contrasts: average peak in LOW
divided by average peak in HIGH, under both peak readings and windows.

In [ ]:
def scaling(d, w, mode):
    vals = bin_values(d, f"peak_{w[0]}_{w[1]}", w, mode, "open", by=("cell", "bin", "contrast"))
    pv = vals.pivot_table(index=["cell", "bin"], columns="contrast", values="value").dropna()
    pv["value"] = pv.LOW / pv.HIGH
    return pv.reset_index()

rows, SCAL = [], {}
for sd in ("time", "trials"):
    cells = INCL[(sd, "both")]
    for wn, w in WINDOWS.items():
        for mode in ("avgmax", "meanmax"):
            v = scaling(ctb[ctb.cell.isin(cells)], w, mode)
            SCAL[(sd, wn, mode)] = v
            H, p, top, means = kw(v)
            med = v.groupby("bin").value.median()
            rows.append({"SD": sd, "window": wn, "peak": mode, "cells": v.cell.nunique(), "cell_bins": len(v),
                         "H": round(H, 2), "p": p, "top_bin(mean)": zp.BIN_LABELS[top], "interior": 0 < top < 6,
                         "HIGH<=0": int((v.HIGH <= 0).sum()),
                         **{f"mean{b}": round(means.get(b, np.nan), 2) for b in range(7)},
                         **{f"med{b}": round(med.get(b, np.nan), 2) for b in range(7)}})
F1H = pd.DataFrame(rows)
print(F1H.to_string())
v = SCAL[("time", "2-2.8", "avgmax")]
print("main reading: range of factors", v.value.min().round(2), "to", v.value.max().round(2))

**Result: reproduced.** The contrast scaling factor differs across bins under all eight readings (Kruskal-Wallis p = 2.5e-17 to 1.1e-09), and the bin with the highest mean is 54-65% in seven readings and 43-54% in one, never the first or last. The means are pulled by a few cells whose HIGH peak is near or below 0 (factors from -109.5 to 37.8 in the main reading; 0 to 5 cell-bins with a HIGH peak at or below 0 per reading); the medians also peak in the 54-65% bin (1.34) in the main reading. The paper's figure has its maximum in the 43-54% bin, which the main reading gives when AA0335 is left out (variants at the end).

In [ ]:
rows = []
for key, v in SCAL.items():
    for b, g in v.groupby("bin"):
        x = g.value.values
        rows.append({"SD": key[0], "window": key[1], "peak": key[2], "bin": zp.BIN_LABELS[b], "n": len(x),
                     "mean": x.mean().round(3), "median": np.median(x).round(3),
                     "wilcoxon_2s": stats.wilcoxon(x - 1).pvalue,
                     "wilcoxon_1s": stats.wilcoxon(x - 1, alternative="greater").pvalue,
                     "t_2s": stats.ttest_1samp(x, 1).pvalue,
                     "t_1s": stats.ttest_1samp(x, 1, alternative="greater").pvalue})
F1HE = pd.DataFrame(rows)
F1HE["all_sig"] = (F1HE[["wilcoxon_2s", "wilcoxon_1s", "t_2s", "t_1s"]] < 0.05).all(1) & (F1HE["mean"] > 1)
print(F1HE.round(4).to_string())
print(F1HE.groupby(["SD", "window", "peak"]).all_sig.sum().rename("bins significant under all four tests (of 7)").to_string())
for test in ["wilcoxon_2s", "wilcoxon_1s", "t_2s", "t_1s"]:
    print(test, "bins with p < 0.05 and mean > 1, by reading:",
          F1HE.assign(ok=(F1HE[test] < 0.05) & (F1HE["mean"] > 1)).groupby(["SD", "window", "peak"]).ok.sum().to_dict())

**Result: not reproduced.** Under no reading are all seven bins significantly above 1. The 32-43% bin has a mean factor of 1.006 to 1.035 and p >= 0.17 under every reading and test. The best readings have six bins significant under some of the tests (for example SD over time, 2-3.2 s, mean of trial maxima, under both Wilcoxon tests and the one-sided t test). Under the main reading the 43-54% bin has a mean of 0.619 (median 1.207) and five bins pass all four tests.

### Two-way ANOVA of peak responses by bin and contrast (Fig. 1i)

In [ ]:
import statsmodels.api as sm
def anova_1i(d, w, mode, pairing):
    vals = bin_values(d, f"peak_{w[0]}_{w[1]}", w, mode, "open", by=("cell", "bin", "contrast"))
    if pairing == "both":
        pv = vals.pivot_table(index=["cell", "bin"], columns="contrast", values="value").dropna()
        keep = set(pv.index)
        vals = vals[[k in keep for k in zip(vals.cell, vals.bin)]]
    m = smf.ols("value ~ C(bin) * C(contrast)", data=vals).fit()
    a = sm.stats.anova_lm(m, typ=2)
    return vals, m, a

rows, ANOVA = [], {}
for (sd, bc), cells in INCL.items():
    for wn, w in WINDOWS.items():
        for mode in ("avgmax", "meanmax"):
            for pairing in ("all", "both"):
                vals, m, a = anova_1i(ctb[ctb.cell.isin(cells)], w, mode, pairing)
                ANOVA[(sd, bc, wn, mode, pairing)] = (vals, m, a)
                rows.append({"SD": sd, "contrasts": bc, "window": wn, "peak": mode, "values_in": pairing,
                             "n_values": len(vals), "df_err": int(m.df_resid),
                             "F_bin": a.loc["C(bin)", "F"].round(2), "p_bin": a.loc["C(bin)", "PR(>F)"],
                             "F_int": a.loc["C(bin):C(contrast)", "F"].round(2),
                             "p_int": a.loc["C(bin):C(contrast)", "PR(>F)"],
                             "F_contrast": a.loc["C(contrast)", "F"].round(2), "p_contrast": a.loc["C(contrast)", "PR(>F)"]})
F1I = pd.DataFrame(rows)
F1I["count_ok"] = (F1I.n_values - 1885).abs() <= 188.5
print(F1I.to_string())
print("paper: 1885 values (error df 1871), F_bin = 10.74, F_int = 6.01")

**Result: reproduced.** The readings that keep only cell-bins with both contrasts give 1830 (SD over time, both contrasts) and 1932 values (contrasts pooled), within the tolerance of 1885. In these, the bin effect is F = 38.07 to 41.12 (p <= 1.0e-43) and the bin by contrast interaction F = 2.21 to 3.02 (p = 0.0061 to 0.040), so both are significant. Keeping every cell-bin-contrast gives 2185 or 2304 values and the interaction F = 4.52 to 4.97 (p <= 0.00015). The bin effect is larger than the paper's F = 10.74 in every reading. Without AA0335 the main reading gives an interaction p of 0.0046; with the single cleaning pass it gives p = 0.17 (variants at the end).

In [ ]:
def posthoc(vals, m):
    mse = m.mse_resid; df = m.df_resid
    g = vals.groupby(["bin", "contrast"]).value.agg(["mean", "count"])
    rows = []
    for b in range(7):
        if (b, "LOW") not in g.index or (b, "HIGH") not in g.index:
            continue
        lo, hi = g.loc[(b, "LOW")], g.loc[(b, "HIGH")]
        tval = (lo["mean"] - hi["mean"]) / np.sqrt(mse * (1 / lo["count"] + 1 / hi["count"]))
        p = 2 * stats.t.sf(abs(tval), df)
        rows.append({"bin": zp.BIN_LABELS[b], "LOW": round(lo["mean"], 2), "HIGH": round(hi["mean"], 2),
                     "t": round(tval, 2), "p_raw": p, "p_bonf7": min(1, p * 7), "p_bonf91": min(1, p * 91)})
    return pd.DataFrame(rows)

rows = []
for key, (vals, m, a) in ANOVA.items():
    ph = posthoc(vals, m)
    for fam in ("p_bonf7", "p_bonf91"):
        sig = ph.loc[ph[fam] < 0.05, "bin"].tolist()
        rows.append({"SD": key[0], "contrasts": key[1], "window": key[2], "peak": key[3], "values_in": key[4],
                     "family": fam, "significant_bins": sig, "only_43_54": sig == ["43-54"],
                     "p_43_54": ph.set_index("bin").loc["43-54", fam]})
F1IP = pd.DataFrame(rows)
print(F1IP.to_string())
allp = F1IP[(F1IP.values_in == "all") & (F1IP.family == "p_bonf91")]
for sd, g in allp.groupby("SD"):
    n = F1I[(F1I.SD == sd) & (F1I.values_in == "all")].n_values.unique().tolist()
    print(f"values_in = all, SD over {sd} ({n} values): 43-54 the only significant bin with 91 comparisons "
          f"in {int(g.only_43_54.sum())} of {len(g)} readings")
key = ("time", "both", "2-2.8", "avgmax", "all")
print("reading with every cell-bin-contrast (SD over time, both contrasts, 2-2.8 s, avgmax; 2185 values, "
      "outside the count tolerance), per bin:")
print(posthoc(*ANOVA[key][:2]).to_string())

**Result: not reproduced.** On the ANOVAs whose count of values is within tolerance (fig1i-baseline-contrast-interaction), the 43-54% bin is not significant under either Bonferroni family (corrected p = 0.095 to 0.22 with seven comparisons, 1 with 91), and the 54-65% bin is the only significant one under both families. Keeping every cell-bin-contrast (2185 or 2304 values, outside the tolerance), the 43-54% bin is significant (main reading LOW 15.54 against HIGH 10.43% dF/F, corrected p = 0.00016 with seven comparisons and 0.0021 with 91); with 91 comparisons it is the only one in 5 of the 8 such readings with SD over time (2185 or 2304 values) and in 8 of 8 with SD across trials (1504 or 1669 values, also outside the tolerance), as the paper reports, while with seven the 54-65% bin is significant too.

## Figure 2: evoked PD in high and low contrast (CGC-assay trials)

Per trial, the mean evoked PD (% baseline) over 0 to 8 s and over 1 to 2 s from DRC onset; trials with
baseline PD < 50% of animal max; CTRL trials of the CGC assay. Mice readings: (a) group A, (b) group B
before ZX1, (c) all mice with CTRL CGC-assay trials (A and B pooled; they share five mice). Unit: (a)
trials, (b) per-mouse means per contrast. Two-sided Wilcoxon rank sum, high against low.

In [ ]:
ptp = tr[(tr.kind == "pt") & (tr.treat == "CTRL")]
MICE2 = {"A": GROUPS["A"], "B": GROUPS["B"], "A+B": sorted(set(GROUPS["A"]) | set(GROUPS["B"]))}
print("mice shared by A and B:", sorted(set(GROUPS["A"]) & set(GROUPS["B"])))
def fig2_test(col):
    rows = []
    for name, mice in MICE2.items():
        d = ptp[ptp.subject.isin(mice) & (ptp.base_pct_max < 50)]
        hi, lo = d[d.contrast == "HIGH"][col], d[d.contrast == "LOW"][col]
        r = stats.ranksums(hi, lo)
        pm = d.groupby(["subject", "contrast"])[col].mean().unstack()
        r2 = stats.ranksums(pm.HIGH.dropna(), pm.LOW.dropna())
        rows.append({"mice": name, "n_mice": d.subject.nunique(), "unit": "trials", "n_HIGH": len(hi), "n_LOW": len(lo),
                     "median_HIGH": hi.median().round(2), "median_LOW": lo.median().round(2),
                     "z": round(r.statistic, 3), "p": r.pvalue})
        rows.append({"mice": name, "n_mice": d.subject.nunique(), "unit": "mouse means", "n_HIGH": pm.HIGH.notna().sum(),
                     "n_LOW": pm.LOW.notna().sum(), "median_HIGH": pm.HIGH.median().round(2),
                     "median_LOW": pm.LOW.median().round(2), "z": round(r2.statistic, 3), "p": r2.pvalue})
    return pd.DataFrame(rows)
F2B = fig2_test("ev_0_8")
print(F2B.to_string())
d = ptp[ptp.subject.isin(GROUPS["A"]) & (ptp.base_pct_max < 50)]
print("group A, mean evoked PD 0-8 s by mouse and contrast (trials < 50%):")
print(d.groupby(["subject", "contrast"]).ev_0_8.agg(["mean", "count"]).unstack().round(1).to_string())
print("range of values:", d.ev_0_8.min().round(1), "to", d.ev_0_8.max().round(1))
print("CGC-assay CTRL trials of group A with a usable pupil:", int(ptp.subject.isin(GROUPS["A"]).sum()),
      "; below 50%:", len(d))
# variant without the 50% limit
dd = ptp[ptp.subject.isin(GROUPS["A"])]
r = stats.ranksums(dd[dd.contrast == "HIGH"].ev_0_8, dd[dd.contrast == "LOW"].ev_0_8)
print(f"variant, group A, all trials: z = {r.statistic:.3f}, p = {r.pvalue:.4f}")

**Result: not reproduced.** Evoked PD over 0 to 8 s is higher in HIGH than LOW under every reading, but not significantly (group A trials, medians 109.10 against 105.05%, z = 1.336, p = 0.18; group B trials p = 0.40; A and B pooled p = 0.054; per-mouse means p = 0.29 to 0.83). Only 82 of 209 group A trials have a baseline below 50% of animal max, from six mice. Without the 50% limit group A gives z = 2.747, p = 0.0060 (a variant). Without AA0335, group A trials give p = 0.028, while B and the pooled trials stay above 0.05 (variants at the end), so the readings would disagree and the criterion would give methods_underspecified: the outcome rests on keeping AA0335 in group A, as the files' label does (an assumption). With the narrower step rule the trial readings give p = 0.088 to 0.54.

In [ ]:
F2C = fig2_test("ev_1_2")
print(F2C.to_string())
dd = ptp[ptp.subject.isin(GROUPS["A"])]
r = stats.ranksums(dd[dd.contrast == "HIGH"].ev_1_2, dd[dd.contrast == "LOW"].ev_1_2)
print(f"variant, group A, all trials: z = {r.statistic:.3f}, p = {r.pvalue:.4f}")

**Result: blocked (methods_underspecified).** The readings disagree: with group A and B mice pooled, trials as units, HIGH exceeds LOW (medians 109.10 against 104.47%, z = 2.558, p = 0.011), while group A alone (p = 0.099), group B alone (p = 0.16) and every per-mouse reading (p = 0.23 to 0.57) are not significant. The criterion asks for agreement and blocks the claim otherwise. Without the 50% limit, group A gives p = 0.065. The readings also disagree without AA0335 (group A trials p = 0.008), under the round-1 cleaning (pooled p = 0.0098), the single pass (pooled p = 0.037) and the narrower step rule (pooled p = 0.010; variants at the end).

### DRC responses of cells before and after ZX1 (Fig. 2d, e; group B)

Per cell and contrast, the mean over 0 to 2 s of the trial-averaged dF/F; tone-responsive cells under
each inclusion reading, determined within each condition (the files segment cells separately before and
after ZX1 in most mice). Two-sided Wilcoxon rank sum, low against high, in CTRL and in ZX1.

In [ ]:
ptB = trials[(trials.kind == "pt") & trials.subject.isin(GROUPS["B"]) & trials.treat.isin(["CTRL", "ZX1"])]
ctB = zp.cell_trial_table(ptB)
print("group B mice:", GROUPS["B"], "; genotypes:", summary.loc[GROUPS["B"], "genotype"].value_counts().to_dict())
print("planes by condition:", ctB.groupby("treat").plane.unique().map(list).to_dict())
rows = []
for sd in ("time", "trials"):
    for both in (True, False):
        for mice_name, mice in (("B", GROUPS["B"]), ("B without ZnT3-KO", [m for m in GROUPS["B"] if summary.loc[m, "genotype"] != "ZnT3KO"]),
                                ("B C57BL6/J only", [m for m in GROUPS["B"] if summary.loc[m, "genotype"] == "C57BL6/J"])):
            for cond in ("CTRL", "ZX1"):
                d = ctB[(ctB.treat == cond) & ctB.subject.isin(mice)]
                cells = zp.responsive(d, (2.0, 2.8), sd=sd, by_contrast=both)
                v = d[d.cell.isin(cells)].groupby(["cell", "contrast"]).drc_0_2.mean().unstack()
                r = stats.ranksums(v.LOW.dropna(), v.HIGH.dropna())
                rows.append({"SD": sd, "contrasts": "both" if both else "pooled", "mice": mice_name, "condition": cond,
                             "cells": len(cells), "median_LOW": v.LOW.median().round(3), "median_HIGH": v.HIGH.median().round(3),
                             "median_diff_LOW-HIGH": (v.LOW - v.HIGH).median().round(3), "z": round(r.statistic, 4), "p": r.pvalue})
F2E = pd.DataFrame(rows)
print(F2E.to_string())
print("paper: 520 cells from 14 mice; CTRL z = 0.0073, p = 0.9942; ZX1 z = -1.2425, p = 0.2141")

**Result: not reproduced.** DRC responses over 0 to 2 s differ between contrasts under every reading, in CTRL and after ZX1: LOW is below HIGH (median cell difference -1.05 to -2.03% dF/F; Wilcoxon rank sum z = -4.94 to -7.57, p = 3.7e-14 to 7.7e-07). Under the main inclusion reading 359 cells in CTRL and 363 after ZX1 from 12 mice, against the paper's 520 from 14; the counts are outside the tolerance under every reading (149 to 375). With the six C57BL6/J mice of group B only, the result is the same (p <= 7.7e-07).

## Figure 3: LME of pure-tone responses on baseline and evoked PD (group A)

Rows are cell by trial (CTRL CGC-assay trials of group A), with the trial's peak pure-tone response
(2-2.8 s window; 2-3.2 s as a variant), its baseline PD (% animal max), contrast, and the trial's mean
evoked PD over 1 to 2 s (% baseline). The Methods' random intercepts for baseline_PD and pupil_at_1-2s
group rows by their values, which are values of the trial, so each is a random intercept per trial; the
two give the same grouping and are fitted as one trial intercept nested in subject. Fitted by maximum
likelihood with statsmodels MixedLM; the fixed effects are tested with Wald z tests (statsmodels has no
Satterthwaite F; the paper's denominator df are about 165, where t and z p-values differ little).

In [ ]:
import statsmodels.regression.mixed_linear_model as mlm
def fit_fig3(d, formula):
    """ML fit; tries several optimizers and keeps the best likelihood (the fits are fragile)."""
    m = smf.mixedlm(formula, d, groups=d["subject"], re_formula="1", vc_formula={"trial": "0 + C(trial)"})
    best = None
    for method in ("lbfgs", "bfgs", "nm", "powell"):
        try:
            r = m.fit(reml=False, method=method, maxiter=2000)
        except Exception:
            continue
        if np.isfinite(r.llf) and (best is None or r.llf > best.llf):
            best = r
    return best

ct3 = ctb.copy()
# predictors centred (baseline at 50%, evoked PD at 100%) to help the optimizer; slopes are unchanged
ct3["pupil12"] = ct3.trial.map(tr.set_index("trial_id").ev_1_2) - 100
ct3["base"] = ct3["base"] - 50
ct3["HIGH"] = (ct3.contrast == "HIGH").astype(float)
rows, FITS3 = [], {}
for cells_name, cells in (("both contrasts, SD time", INCL[("time", "both")]), ("all tone-responsive, SD time", INCL[("time", "pooled")]),
                          ("both contrasts, SD trials", INCL[("trials", "both")]), ("all tone-responsive, SD trials", INCL[("trials", "pooled")])):
    for trials_name in ("all", "<50%"):
        for wn, w in WINDOWS.items():
            d = ct3[ct3.cell.isin(cells)]
            if trials_name == "<50%":
                d = d[d.base < 0]  # base is centred at 50%
            d = d.rename(columns={f"peak_{w[0]}_{w[1]}": "peak"})[["peak", "base", "HIGH", "pupil12", "subject", "trial"]].dropna()
            full = fit_fig3(d, "peak ~ base + HIGH * pupil12")
            null = fit_fig3(d, "peak ~ base")
            FITS3[(cells_name, trials_name, wn)] = (full, null, d)
            lr = 2 * (full.llf - null.llf)
            rows.append({"cells": cells_name, "trials": trials_name, "window": wn, "rows": len(d),
                         "n_cells": d.shape[0] and ctb[ctb.cell.isin(cells)].cell.nunique(), "n_trials": d.trial.nunique(),
                         "b_base": full.params["base"], "p_base": full.pvalues["base"],
                         "b_pupil12": full.params["pupil12"], "p_pupil12": full.pvalues["pupil12"],
                         "b_HIGHxpupil12": full.params["HIGH:pupil12"], "p_HIGHxpupil12": full.pvalues["HIGH:pupil12"],
                         "b_HIGH": full.params["HIGH"], "p_HIGH": full.pvalues["HIGH"],
                         "var_subject": full.cov_re.iloc[0, 0], "var_trial": full.vcomp[0], "var_resid": full.scale,
                         "LR": lr, "LR_df": 3, "p_LRT": stats.chi2.sf(lr, 3), "converged": full.converged and null.converged})
F3 = pd.DataFrame(rows)
print(F3.to_string())
print("paper: baseline PD p = 3.64e-3; evoked PD p = 0.108; interaction p = 0.756; LRT p = 0.330")

**Result: reproduced.** In all sixteen fits (four cell inclusion readings, two trial readings, two peak windows) the baseline PD coefficient is negative and significant (b = -0.100 to -0.191% dF/F per % of max, p = 1.2e-04 to 0.032), while evoked PD at 1-2 s (p = 0.42 to 0.92) and its interaction with contrast (p = 0.22 to 0.72) are not, as the paper reports. All fits converged; the subject and trial variances are printed. The paper does not give the sign of the baseline coefficient. Without AA0335, the baseline coefficient is not significant in the trials below 50% (p = 0.087, one fit, variants at the end), which would make the outcome partially reproduced: the outcome rests on keeping AA0335 in group A, as the files' label does (an assumption).

In [ ]:
print(F3[["cells", "trials", "window", "rows", "LR", "LR_df", "p_LRT", "b_HIGH", "p_HIGH"]].to_string())
print("paper: likelihood ratio test p = 0.330")

**Result: not reproduced.** With all trials, the likelihood ratio test of the full model against the null model is significant in all eight fits (chi-squared on 3 df, p = 0.0013 to 0.015), against the paper's p = 0.330; with the trials below 50% it gives p = 0.041 to 0.17. The criterion asks for p >= 0.05 under each trial reading. The full model also carries the contrast main effect, which is significant in some all-trial fits (b = -1.41 to -1.61, p = 0.030 to 0.060 with SD over time) and accounts for part of the difference.

## Figure 4: growth curve analysis of evoked PD in ZnT3-WT and ZnT3-KO mice

DRC trials without the tone (`contrastChange`), CTRL and ZX1, baseline PD < 50% of animal max; pupil
(% baseline) at the 21 samples from 0 to 2 s; orthogonal time polynomials with unit-norm columns over
those samples (R `poly()`). LME with a random intercept per mouse, fitted by REML (statsmodels MixedLM;
lmer's default) with several optimizers, keeping the converged fit of highest likelihood (`zp.fit_lme`;
lbfgs alone stopped at a degenerate point with an infinite likelihood and a zero intercept in some fits). The
optimizer, convergence, subject variance and whether it is on the boundary (0) are printed for each fit. Readings: model (a) the written formula plus contrast:treatment, (b) the full factorial;
coding (a) reference (LOW, CTRL), (b) sum coding (±1). Terms tested with Wald z (the paper's
Satterthwaite F have about 4500 to 7900 denominator df, where z and t agree).

In [ ]:
drc = tr[(tr.kind == "drc")].copy()
drc["is_high"] = (drc.contrast == "HIGH").astype(float)
FORMS = {"written+cxt": "pupil ~ (tPoly1 + tPoly2) * c + (tPoly1 + tPoly2) * x + c:x",
         "full factorial": "pupil ~ (tPoly1 + tPoly2) * c * x"}
def code(long, factor_col, level, coding):
    v = (long[factor_col] == level).astype(float)
    return v if coding == "reference" else 2 * v - 1

def gca(trs, factor="treat", level="ZX1", forms=FORMS, codings=("reference", "sum")):
    long = zp.gca_long(trs)
    long["c"] = 0.0
    out, fits = [], {}
    for coding in codings:
        long["c"] = code(long, "contrast", "HIGH", coding)
        long["x"] = code(long, factor, level, coding)
        for fname, f in forms.items():
            m, diag = zp.fit_lme(f, long, "subject", reml=True)
            fits[(fname, coding)] = (m, long.copy())
            row = {"model": fname, "coding": coding, "rows": len(long), "trials": long.trial.nunique(),
                   "mice": long.subject.nunique(), **diag, "intercept": m.params["Intercept"]}
            for term, lab in (("c", "contrast_int"), ("tPoly1:c", "contrast_slope"), ("x", "x_int"),
                              ("tPoly1:x", "x_slope"), ("c:x", "c_x")):
                if term in m.params:
                    row[f"{lab}_b"] = m.params[term]; row[f"{lab}_z"] = m.tvalues[term]; row[f"{lab}_p"] = m.pvalues[term]
            out.append(row)
    return pd.DataFrame(out), fits, long

GC, GD = GROUPS["C"], GROUPS["D"]  # not C: patsy formulas use C()
wt = drc[drc.subject.isin(GC) & drc.treat.isin(["CTRL", "ZX1"]) & (drc.base_pct_max < 50)]
print("ZnT3-WT DRC trials with a usable pupil:", int((drc.subject.isin(GC)).sum()), "; below 50%:", len(wt))
print(pd.crosstab([wt.subject], [wt.treat, wt.contrast]).to_string())
F4C, FITS_WT, long_wt = gca(wt)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
print(F4C.to_string())
print("paper (WT): contrast intercept F = 19.48 p = 1.03e-5; contrast slope F = 26.54 p = 2.64e-7;"
      " ZX1 intercept F = 4.47 p = 0.0344; ZX1 slope F = 40.45 p = 2.13e-10; contrast x ZX1 F = 24.2 p = 8.85e-7; df about 7920")
terms = ["contrast_int_p", "contrast_slope_p", "x_int_p", "x_slope_p", "c_x_p"]
print("terms with p < 0.05 per reading:", (F4C[terms] < 0.05).sum(1).tolist())

In [ ]:
# mean traces (0-2 s) by condition, and per-mouse mean evoked PD over 0-2 s
mt = long_wt.groupby(["treat", "contrast", "time"]).pupil.mean().unstack(["treat", "contrast"])
print(mt.iloc[::4].round(2).to_string())
print(wt.groupby(["subject", "treat", "contrast"]).ev_0_2.mean().unstack(["treat", "contrast"]).round(1).to_string())
fig, ax = plt.subplots(figsize=(4, 3))
for c in mt.columns:
    ax.plot(mt.index, mt[c], label=" ".join(c))
ax.legend(); ax.set(xlabel="time (s)", ylabel="pupil (% baseline)", title="ZnT3-WT"); plt.tight_layout(); plt.show()
print("plotted range", np.nanmin(mt.values).round(1), "to", np.nanmax(mt.values).round(1))

**Result: partially reproduced.** 328 ZnT3-WT trials below 50% (6888 rows, against the paper's denominator df of about 7920). Under reference coding all five terms are significant in both models: contrast intercept (b = 2.48, z = 5.81, p = 6.3e-09), contrast slope (p = 0.049 written model, 0.0029 full factorial), ZX1 intercept (b = 1.25, p = 0.010), ZX1 slope (b = -8.24 and -4.96, p = 4.0e-08 and 0.017) and contrast by ZX1 (b = -2.76, z = -4.18, p = 3.0e-05). Under sum coding the ZX1 intercept is not significant (p = 0.72) and, in the full factorial, nor is the contrast slope (p = 0.12), so four and three terms pass. Every reading meets the partial condition (the interaction and at least two main terms; 4 of 4 in the variants table), and two also meet the full one. The readings thus agree on partially reproduced and differ only on whether the full condition also holds, so the criterion's partial clause, read as applying under every reading as its full clause does, gives partially reproduced. Reference coding is the default of both fitlme and lmer. The mean traces show the expected order in CTRL (HIGH 113.2 against LOW 108.5% at 2 s) and a reversed one after ZX1 (106.9 against 109.1). All fits converged with a subject variance away from 0.

The outcome rests on the cleaning (an assumption): the partial condition holds under all four readings with the main cleaning, with the narrower step rule (three or four terms, interaction p <= 0.00093) and with the round-1 cleaning without AA0319 and AA0334, and under none with the round-1 cleaning (interaction p = 0.25) or the single pass (p = 0.56; variants at the end). This revision had kept the outcome blocked and proposed partially reproduced; the third verification round agreed, and the outcome was changed.

In [ ]:
def emmeans_tukey(m, long, coding):
    """Cell means at tPoly1 = tPoly2 = 0 for the 4 contrast x treatment cells, Tukey-adjusted pairs."""
    names = m.fe_params.index
    cov = m.cov_params().loc[names, names].values
    lv = {"reference": (0.0, 1.0), "sum": (-1.0, 1.0)}[coding]
    cells = {}
    for cn, cv in (("LOW", lv[0]), ("HIGH", lv[1])):
        for xn, xv in (("CTRL", lv[0]), ("X", lv[1])):
            L = np.zeros(len(names))
            for i, n in enumerate(names):
                if n == "Intercept": L[i] = 1
                elif n == "c": L[i] = cv
                elif n == "x": L[i] = xv
                elif n == "c:x": L[i] = cv * xv
            cells[(cn, xn)] = L
    df_ = len(long) - len(names)
    rows = []
    keys = list(cells)
    for i in range(4):
        for j in range(i + 1, 4):
            L = cells[keys[i]] - cells[keys[j]]
            est = L @ m.fe_params.values
            se = np.sqrt(L @ cov @ L)
            tval = est / se
            p = stats.studentized_range.sf(abs(tval) * np.sqrt(2), 4, df_)
            rows.append({"pair": f"{keys[i][0]} {keys[i][1]} - {keys[j][0]} {keys[j][1]}", "estimate": est, "z": tval, "p_tukey": p})
    return pd.DataFrame(rows), {k: v @ m.fe_params.values for k, v in cells.items()}

def posthoc_summary(fits):
    rows = []
    for key, (m, long) in fits.items():
        ph, means = emmeans_tukey(m, long, key[1])
        p = ph.set_index("pair").p_tukey; e = ph.set_index("pair").estimate
        rows.append({"model": key[0], "coding": key[1],
                     "HIGH-LOW CTRL": e["LOW CTRL - HIGH CTRL"] * -1, "p": p["LOW CTRL - HIGH CTRL"],
                     "HIGH-LOW X": e["LOW X - HIGH X"] * -1, "p ": p["LOW X - HIGH X"],
                     "X-CTRL HIGH": e["HIGH CTRL - HIGH X"] * -1, "p  ": p["HIGH CTRL - HIGH X"],
                     "X-CTRL LOW": e["LOW CTRL - LOW X"] * -1, "p   ": p["LOW CTRL - LOW X"],
                     **{f"mean {k[0]} {k[1]}": v for k, v in means.items()}})
    return pd.DataFrame(rows)
PH_WT = posthoc_summary(FITS_WT)
print(PH_WT.to_string())
ok = ((PH_WT["HIGH-LOW CTRL"] > 0) & (PH_WT["p"] < 0.05) & ~(PH_WT["p "] < 0.05) & (PH_WT["X-CTRL HIGH"] < 0)
      & (PH_WT["p  "] < 0.05) & ~(PH_WT["p   "] < 0.05))
print("criterion met per reading:", ok.tolist())
print(emmeans_tukey(*FITS_WT[("written+cxt", "reference")], "reference")[0].to_string())

**Result: not reproduced.** At the cell means, HIGH exceeds LOW in CTRL by 2.48% (Tukey p = 3.9e-08) and not after ZX1 (-0.28, p = 0.95), and ZX1 lowers HIGH (-1.51, p = 0.015), as the paper reports, but ZX1 also raises LOW (1.25, p = 0.0496), which the criterion requires not to be significant, and the ZX1 effect on HIGH is weaker than the paper's p < 0.0001. Three of the four parts hold; the fourth fails at the threshold, and the criterion has no partial level, so it is not met. The same numbers come out under all four model and coding readings.

The result also turns with the cleaning (variants table, written model with reference coding): with the round-1 cleaning HIGH minus LOW in CTRL is not significant (p = 0.078) and ZX1 changes neither contrast (p = 0.83 and 0.89); with the round-1 cleaning without AA0319 and AA0334 ZX1 changes LOW (p = 0.00011) and not HIGH (p = 0.31), the reverse of the paper; with the single pass HIGH and LOW differ after ZX1 too (p = 0.0045). Only with the narrower step rule are all four parts met (p = 1.7e-05, 0.98, 0.00036 and 0.99). The outcome therefore rests on the main cleaning (an assumption) and on a p-value at the threshold. This revision had kept the outcome partially reproduced and proposed not reproduced; the third verification round agreed, and the outcome was changed.

In [ ]:
ko = drc[drc.subject.isin(GD) & drc.treat.isin(["CTRL", "ZX1"]) & (drc.base_pct_max < 50)]
print("ZnT3-KO DRC trials with a usable pupil:", int((drc.subject.isin(GD)).sum()), "; below 50%:", len(ko))
print(pd.crosstab([ko.subject], [ko.treat, ko.contrast]).to_string())
F4F, FITS_KO, long_ko = gca(ko)
print(F4F.to_string())
print("paper (KO): contrast intercept p = 0.255; ZX1 intercept p = 0.147; contrast x ZX1 p = 0.589;"
      " contrast slope p = 0.190; ZX1 slope p = 0.0129 (weight -0.64); df about 4500")
absent = ["contrast_int_p", "x_int_p", "c_x_p", "contrast_slope_p"]
print("absence terms all p >= 0.05 per reading:", (F4F[absent] >= 0.05).all(1).tolist())
mt = long_ko.groupby(["treat", "contrast", "time"]).pupil.mean().unstack(["treat", "contrast"])
print(mt.iloc[::4].round(2).to_string())

**Result: blocked (methods_underspecified).** 176 ZnT3-KO trials below 50% (3696 rows, against about 4500 in the paper). The contrast intercept (p = 0.32 to 0.74), ZX1 intercept (p = 0.26 to 0.30) and contrast by ZX1 (p = 0.58) terms are not significant under any reading, but the contrast slope is significant in three of four readings (z = 2.60 to 2.66, p = 0.0078 to 0.0095) and not under the full factorial with reference coding (p = 0.18). The criterion treats a disagreement between model readings as methods_underspecified.

In [ ]:
cmp = pd.DataFrame({"model": F4F.model, "coding": F4F.coding,
                    "KO ZX1 slope b": F4F.x_slope_b, "KO p": F4F.x_slope_p, "WT ZX1 slope b": F4C.x_slope_b, "WT p": F4C.x_slope_p})
cmp["KO < half WT"] = cmp["KO ZX1 slope b"].abs() < 0.5 * cmp["WT ZX1 slope b"].abs()
cmp["KO in -0.64±0.064"] = (cmp["KO ZX1 slope b"] + 0.64).abs() <= 0.064
cmp["WT in -1.41±0.141"] = (cmp["WT ZX1 slope b"] + 1.41).abs() <= 0.141
cmp["criterion met"] = (cmp["KO ZX1 slope b"] < 0) & (cmp["KO p"] < 0.05) & cmp["KO < half WT"]
print(cmp.to_string())
print("readings meeting the criterion (KO term negative with p < 0.05 and KO weight < half WT):", int(cmp["criterion met"].sum()), "of", len(cmp))

**Result: not reproduced.** The criterion is met when, under one model and coding reading, the KO ZX1 slope term is negative with p < 0.05 and the KO weight is less than half the WT weight in magnitude. It is evaluated under each of the four readings, and none meets it: the KO term is negative but not significant under any (b = -0.85 to -3.18, p = 0.20 to 0.33). The KO weight is less than half the WT weight in three readings (for example -1.70 against -8.24, written model with reference coding). Neither weight is near the expected values (-0.64 and -1.41): with unit-norm polynomials over the 21 samples the WT weight is -4.12 to -8.24, so the paper's polynomial scaling is not this one. The disagreement that blocks fig4f-ko-no-contrast-effect does not bear on this claim, since its criterion fails under every reading.

## Figure 5

The three Figure 5 claims (fig5-contralateral-zx1-increases, fig5-ipsilateral-zx1-reduces,
fig5c-no-dye-in-other-hemisphere) need the widefield imaging of right A1 in three mice with a cannula in
the left auditory cortex, before and after contralateral and ipsilateral ZX1, and the red-channel images.
All 30 files hold two-photon imaging and pupillometry (above); no file holds widefield imaging or
cannula experiments, so these claims are blocked (data_missing).

## Figure 6: baseline PD drift, ACSF control, baseline against evoked PD, and genotype

### Baseline PD against recording time (Fig. 6a)

Per trial (CGC-assay and DRC trials, all treatments), baseline PD in % animal max against the time of
sound onset in minutes from the experiment's first trial; ordinary least squares pooled over mice. ZX1
readings: (a) groups C and D, (b) group C, (c) groups B, C and D, and (d) the eleven mice the files label
`ZX1_uBasePupilPctMax_wEpoch`. ACSF: group F (the files' `ACSF_uBasePupilPctMax_wEpoch` label names the
same four mice).

In [ ]:
snd_tr = tr[tr.kind.isin(["pt", "drc"])]
def drift(mice, label):
    d = snd_tr[snd_tr.subject.isin(mice)]
    r = stats.linregress(d.rec_min, d.base_pct_max)
    tcrit = stats.t.ppf(0.975, len(d) - 2)
    return {"set": label, "mice": d.subject.nunique(), "trials": len(d), "slope": r.slope,
            "ci_low": r.slope - tcrit * r.stderr, "ci_high": r.slope + tcrit * r.stderr, "p": r.pvalue,
            "minutes": f"{d.rec_min.min():.0f} to {d.rec_min.max():.0f}"}
rows = [drift(GC + GD, "ZX1 (a) C+D"), drift(GC, "ZX1 (b) C"), drift(sorted(set(GROUPS["B"]) | set(GC) | set(GD)), "ZX1 (c) B+C+D"),
        drift(GROUPS["ZX1_drift"], "ZX1 (d) labelled"), drift(GROUPS["F"], "ACSF F")]
F6A = pd.DataFrame(rows)
print(F6A.to_string())
acsf = F6A.iloc[-1]
for _, r in F6A.iloc[:-1].iterrows():
    print(f"{r['set']}: slope > 0 and p < 0.05: {r.slope > 0 and r.p < 0.05}; CI overlaps ACSF CI: "
          f"{max(r.ci_low, acsf.ci_low) <= min(r.ci_high, acsf.ci_high)}")
print("ACSF: slope > 0 and p < 0.05:", acsf.slope > 0 and acsf.p < 0.05)
print("paper: ZX1 CI (0.0798, 0.183), p = 1.26e-6; ACSF CI (0.0159, 0.279), p = 0.0282")
print("per-mouse slopes (% per min):", snd_tr.groupby("subject").apply(lambda g: stats.linregress(g.rec_min, g.base_pct_max).slope).round(3).to_dict())

**Result: partially reproduced.** Baseline PD rises with recording time in both experiments: under the four ZX1 readings, slopes of 0.131 to 0.229% of max per minute (95% CIs within 0.107 to 0.255, p <= 7.7e-19), and in the ACSF mice 0.351 (CI 0.304 to 0.398, p = 4.3e-35). But the ACSF interval does not overlap any ZX1 interval, which the criterion requires; the paper's ACSF interval (0.0159 to 0.279) is wider and lower, while the intervals of ZX1 readings (a) and (b) (0.107 to 0.155 and 0.111 to 0.171) lie inside the paper's ZX1 interval (0.0798 to 0.183). The drift is present in both, as the paper reports, and the claim that the two are alike does not hold. Per-mouse slopes range from -0.349 to 0.776. No cleaning variant gives an overlap under readings (a) to (c), which the criterion lists; with the round-1 cleaning without AA0319 and AA0334 the added reading (d) overlaps (0.231 to 0.285 against ACSF 0.281 to 0.371; variants at the end).

### Contrast effect after ACSF (Fig. 6b)

Group F, DRC trials, baseline PD < 50%. Readings: (a) ACSF trials only, pupil ~ (tPoly1 + tPoly2) x
contrast + (1 | subject); (b) trials before and after ACSF, the models of Fig. 4 with ACSF as treatment,
and the contrast effect within ACSF from the cell means.

In [ ]:
F_ = GROUPS["F"]
ac = drc[drc.subject.isin(F_) & (drc.base_pct_max < 50)]
print("group F DRC trials with a usable pupil:", int(drc.subject.isin(F_).sum()), "; below 50%:", len(ac))
print(pd.crosstab([ac.subject], [ac.treat, ac.contrast]).to_string())
ra, _, long_a = gca(ac[ac.treat == "ACSF"], forms={"post only": "pupil ~ (tPoly1 + tPoly2) * c"})
print("(a) ACSF trials only")
print(ra[["model", "coding", "rows", "trials", "mice", "contrast_int_b", "contrast_int_p", "contrast_slope_b", "contrast_slope_p", "converged", "group_var"]].to_string())
rb, fits_b, long_b = gca(ac[ac.treat.isin(["CTRL", "ACSF"])], factor="treat", level="ACSF")
print("(b) before and after ACSF")
print(rb.to_string())
phb = posthoc_summary(fits_b)
print(phb[["model", "coding", "HIGH-LOW CTRL", "p", "HIGH-LOW X", "p "]].rename(columns={"HIGH-LOW X": "HIGH-LOW ACSF", "p ": "p ACSF"}).to_string())
print("paper: contrast intercept F(1,2558.3) = 46.22 p = 1.31e-11; slope F(1,2558) = 15.076 p = 1.10e-4")
mt = long_b.groupby(["treat", "contrast", "time"]).pupil.mean().unstack(["treat", "contrast"])
print(mt.iloc[::4].round(2).to_string())

**Result: partially reproduced.** The contrast intercept effect holds after ACSF under every reading (ACSF trials only, b = 4.40% with HIGH above LOW, p = 1.5e-05; trials before and after, contrast term p = 3.7e-05 with reference and 9.0e-10 with sum coding, and HIGH minus LOW within ACSF 4.17 at the cell means, Tukey p = 2.6e-05). The contrast slope effect does not hold under any reading (p = 0.11 with ACSF trials only, 0.066 to 0.51 with both). Only 33 ACSF trials (693 rows) and 100 trials before and after (2100 rows, against about 2560 in the paper) are below 50%.

### Baseline PD against evoked PD (Fig. 6c, and the correlation in the Methods)

Per trial, baseline PD (% animal max) against the mean evoked PD over 1 to 2 s (% baseline), both
contrasts. The readings the claims file lists: groups A, B before ZX1 (CGC-assay trials), C, D, F (DRC
trials), CTRL or all trials (`in_claims_file`). Two kinds of reading are added here and marked: group B
with its trials after ZX1, and the mice the files label `pupilBaseline_vs_evoked` (DRC, CGC-assay or both
kinds of trial). They are shown, but the outcome is decided on the listed readings, as the criterion says.


In [ ]:
rows = []
kinds = {"A": ["pt"], "B": ["pt"], "C": ["drc"], "D": ["drc"], "F": ["drc"], "base_vs_evoked": ["drc", "pt"]}
for g, ks in kinds.items():
    for kind in ([ks] if len(ks) == 1 else [["drc"], ["pt"], ["drc", "pt"]]):
        for trt in ("CTRL", "all"):
            d = snd_tr[snd_tr.subject.isin(GROUPS[g]) & snd_tr.kind.isin(kind)]
            if trt == "CTRL":
                d = d[d.treat == "CTRL"]
            r_all = stats.pearsonr(d.base_pct_max, d.ev_1_2)
            d50 = d[d.base_pct_max < 50]
            r50 = stats.pearsonr(d50.base_pct_max, d50.ev_1_2)
            # the claims file lists A, B before ZX1, C, D and F, CTRL or all trials; B with its trials
            # after ZX1 and the files' pupilBaseline_vs_evoked mice are readings added here
            listed = g in ("A", "C", "D", "F") or (g == "B" and trt == "CTRL")
            rows.append({"group": g, "kind": "+".join(kind), "trials": trt, "in_claims_file": listed, "n_all": len(d), "r_all": r_all.statistic,
                         "p_all": r_all.pvalue, "n_below50": len(d50), "r_below50": r50.statistic, "p_below50": r50.pvalue})
F6C = pd.DataFrame(rows)
F6C["n50_matches_240"] = (F6C.n_below50 - 240).abs() <= 24
print(F6C.to_string())
listed = F6C[F6C.in_claims_file]
print("claims-file readings within 240 ± 24:", listed[listed.n50_matches_240][["group", "trials", "n_below50", "r_below50", "p_below50"]].to_dict("records"))
print("added readings within 240 ± 24:", F6C[~F6C.in_claims_file & F6C.n50_matches_240][["group", "kind", "trials", "n_below50", "r_below50", "p_below50"]].to_dict("records"))
print("paper (Fig. 6c): r(238) = -0.0928, p = 0.1537, 240 responses")

**Result: blocked (methods_underspecified).** No reading that the claims file lists (groups A, B before ZX1, C, D, F; CTRL or all trials) has a count of responses below 50% within 216 to 264 (they range from 67 to 328), so the criterion shows every result and blocks the claim unless all agree. They do not: group D, all trials, gives a significant correlation (r = -0.273, p = 0.00025), and the other eight do not (p = 0.076 to 0.70). One reading added here, all CGC-assay trials of group B, before and after ZX1, matches the count (230 responses) and gives r = 0.0096, p = 0.88, not significant, as the paper reports (r = -0.0928, p = 0.1537); the claims file lists group B before ZX1 only. The count, and so which reading is selected, depends on the cleaning: with the round-1 cleaning group B, all trials, has 262 responses (p = 0.072), and with the single pass group C, CTRL (250, p = 0.032) and group D, all trials (233, p = 0.00095) match, which would give not reproduced; with the narrower step rule no listed reading matches and one of nine is significant, as with the main cleaning (variants at the end). The outcome rests on the cleaning.

In [ ]:
F6C["n_matches_626"] = (F6C.n_all - 626).abs() <= 62.6
F6C["n50_matches_384"] = (F6C.n_below50 - 384).abs() <= 38.4
print(F6C[["group", "kind", "trials", "in_claims_file", "n_all", "r_all", "p_all", "n_below50", "r_below50", "p_below50", "n_matches_626", "n50_matches_384"]].to_string())
print("paper (Methods): r(624) = -0.35, p = 1.20e-19 over 626 responses; r(382) = -0.055, p = 0.285 below 50%")
# Variants: evoked PD over 0-2 s and 0-8 s, for the readings whose counts match
for _, r in F6C[F6C.n_matches_626 & F6C.n50_matches_384].iterrows():
    d = snd_tr[snd_tr.subject.isin(GROUPS[r.group]) & snd_tr.kind.isin(r.kind.split("+"))]
    if r.trials == "CTRL":
        d = d[d.treat == "CTRL"]
    for col in ("ev_0_2", "ev_0_8"):
        a = stats.pearsonr(d.base_pct_max, d[col]); b = stats.pearsonr(d[d.base_pct_max < 50].base_pct_max, d[d.base_pct_max < 50][col])
        print(f"variant {r.group} {r.kind} {r.trials} {col}: all r = {a.statistic:.3f} p = {a.pvalue:.3g}; below 50% r = {b.statistic:.3f} p = {b.pvalue:.3g}")

**Result: blocked (methods_underspecified).** No reading matches both counts (626 ± 62.6 and 384 ± 38.4), whether the claims file lists it or not; the closest, the files' `pupilBaseline_vs_evoked` mice with DRC trials before and after ZX1, has 563 responses (just under 563.4) and 330 below 50%. The readings do not agree. Over all responses r ranges from -0.213 to -0.422, inside the tolerance (-0.315 to -0.385) for groups A and F (all trials) only, and below 50% group D (all trials) is significant (p = 0.00025) while the others are not. Under no cleaning variant does a reading match both counts (variants at the end).

### Genotype effect on high-contrast responses before ZX1 (Fig. 6d)

In [ ]:
gd = drc[drc.subject.isin(GC + GD) & (drc.treat == "CTRL") & (drc.contrast == "HIGH") & (drc.base_pct_max < 50)].copy()
gd["genotype"] = gd.subject.map(summary.genotype)
r6d, fits6d, long6d = gca(gd, factor="genotype", level="ZnT3KO", forms={"genotype": "pupil ~ (tPoly1 + tPoly2) * x"})
print(r6d[["model", "coding", "rows", "trials", "mice", "x_int_b", "x_int_p", "x_slope_b", "x_slope_z", "x_slope_p", "optimizer", "converged", "group_var", "boundary"]].to_string())
print("paper: genotype on slope F(1,3869) = 22.37, p = 2.33e-6 (about 3870 rows)")
mt = long6d.groupby(["genotype", "time"]).pupil.mean().unstack("genotype")
print(mt.iloc[::4].round(2).to_string())

**Result: reproduced.** 134 CTRL high-contrast trials of 16 ZnT3-WT and ZnT3-KO mice (2814 rows, against about 3870). The genotype by tPoly1 term is negative, so the KO slope is below the WT slope (b = -7.51 with reference coding, z = -3.47, p = 0.00052; the same test under sum coding). The genotype intercept term is not significant (p = 0.39). The mean KO trace rises faster at first and is below WT from 1.2 s (111.0 against 112.5 at 1.2 s, 109.8 against 113.2 at 2 s). Both fits converged with a subject variance of 45.4, away from 0. With the single cleaning pass the term is not significant (b = -3.24, p = 0.20; variants at the end).

### Peak of high-contrast responses (Discussion; Fig. 4c, d)

In [ ]:
def peak_rise(trs):
    long = zp.gca_long(trs)
    m = long.groupby("time").pupil.mean()
    return m.max() - 100, m.idxmax(), long.trial.nunique()
rows = []
for name, d in (("WT after ZX1, HIGH", wt[(wt.treat == "ZX1") & (wt.contrast == "HIGH")]),
                ("WT CTRL, HIGH", wt[(wt.treat == "CTRL") & (wt.contrast == "HIGH")]),
                ("KO CTRL, HIGH (a)", ko[(ko.treat == "CTRL") & (ko.contrast == "HIGH")]),
                ("KO CTRL+ZX1, HIGH (b)", ko[ko.contrast == "HIGH"]),
                ("WT after ZX1, LOW", wt[(wt.treat == "ZX1") & (wt.contrast == "LOW")]),
                ("WT CTRL, LOW", wt[(wt.treat == "CTRL") & (wt.contrast == "LOW")])):
    pk, tpk, n = peak_rise(d)
    rows.append({"set": name, "trials": n, "peak_rise_%": round(pk, 2), "at_s": tpk, "within_10±2.5": abs(pk - 10) <= 2.5})
print(pd.DataFrame(rows).to_string())

**Result: reproduced.** The mean high-contrast trace of ZnT3-WT mice after ZX1 peaks 10.49% above baseline (67 trials), and that of ZnT3-KO mice 11.17% in CTRL (47 trials) and 10.93% with CTRL and ZX1 pooled, all within 10 ± 2.5. The WT CTRL high-contrast peak (13.23%) is outside the interval, but the WT low-contrast peaks (9.76 and 10.01%) are inside it, so the measure separates high from low contrast only in part. The second part of the statement, that both contrasts resemble low-contrast responses, is not tested by the criterion; fig4-wt-posthoc covers it for WT and was not reproduced.

## Variants: pupil cleaning and group A

The pupil claims rest on the cleaning assumption (Data check). Here the number each pupil claim's outcome
rests on is recomputed under each cleaning: the main cleaning (trials with a step by more than a factor 2
between tracked frames dropped), the round-1 cleaning (no trial dropped for steps, so the tracking-failure
runs of AA0319 and AA0334 set their maxima), the round-1 cleaning without those two mice, and the
Methods' single pass (no spike filter, no trial dropped; several animals' "maximum" PD is a tracking
failure). The last column keeps the main cleaning and leaves the ZnT3-WT mouse AA0335 out of group A, so
that group A holds only C57BL6/J mice (nine, against the paper's eight). The last variant narrows the
step rule: steps are counted on the trace after the spike filter, so trials whose only fault is a
spike the filter removes are kept. Model readings: Figure 3, cells
responsive in both contrasts (SD over time), 2-2.8 s; Figure 4 post hoc, written model with reference
coding; Figure 6b, reading (a); Figure 6d, reference coding.


In [ ]:
def key_results(trx, A_mice=None):
    """The number each pupil claim's outcome rests on, recomputed from a table of cleaned trials."""
    A_mice = GROUPS["A"] if A_mice is None else A_mice
    out = {}
    # Figure 1: main reading (SD over time, both contrasts, 2-2.8 s, maximum of the averaged trace)
    c1 = ct[ct.subject.isin(A_mice)].copy()
    c1["base"] = c1.trial.map(trx.set_index("trial_id").base_pct_max)
    c1 = c1[c1.base.notna() & c1.cell.isin(INCL[("time", "both")])]
    v = bin_values(c1, "peak_2.0_2.8", (2.0, 2.8), "avgmax", "open")
    H, p, top, _ = kw(v)
    out["fig1f KW p; top bin; cells"] = f"{p:.2g}; {zp.BIN_LABELS[top]}; {v.cell.nunique()}"
    v = reliability(c1, "pearson")
    H, p, top, _ = kw(v)
    out["fig1g KW p; top bin"] = f"{p:.2g}; {zp.BIN_LABELS[top]}"
    v = scaling(c1, (2.0, 2.8), "avgmax")
    H, p, top, _ = kw(v)
    nsig = sum((g.value.mean() > 1) and stats.wilcoxon(g.value - 1).pvalue < 0.05 for _, g in v.groupby("bin"))
    out["fig1h KW p; top bin (mean); bins above 1 (Wilcoxon)"] = f"{p:.2g}; {zp.BIN_LABELS[top]}; {nsig} of 7"
    vals, m, a = anova_1i(c1, (2.0, 2.8), "avgmax", "both")
    ph = posthoc(vals, m)
    out["fig1i values; p_bin; p_int"] = f"{len(vals)}; {a.loc['C(bin)', 'PR(>F)']:.2g}; {a.loc['C(bin):C(contrast)', 'PR(>F)']:.2g}"
    out["fig1i bins significant (Bonferroni 7; 91)"] = f"{ph.loc[ph.p_bonf7 < 0.05, 'bin'].tolist()}; {ph.loc[ph.p_bonf91 < 0.05, 'bin'].tolist()}"
    # Figure 2: Wilcoxon rank sum, trials as units
    p2 = trx[(trx.kind == "pt") & (trx.treat == "CTRL") & (trx.base_pct_max < 50)]
    for col, name in (("ev_0_8", "fig2b"), ("ev_1_2", "fig2c")):
        ps = []
        for mn, mice in (("A", A_mice), ("B", GROUPS["B"]), ("A+B", sorted(set(A_mice) | set(GROUPS["B"])))):
            d = p2[p2.subject.isin(mice)]
            ps.append(f"{mn} {stats.ranksums(d[d.contrast == 'HIGH'][col], d[d.contrast == 'LOW'][col]).pvalue:.2g}")
        out[f"{name} p, trials"] = "; ".join(ps)
    # Figure 3: cells responsive in both contrasts (SD over time), 2-2.8 s
    for trials_name in ("all", "<50%"):
        d = c1.copy()
        d["pupil12"] = d.trial.map(trx.set_index("trial_id").ev_1_2) - 100
        d["base"] = d["base"] - 50
        d["HIGH"] = (d.contrast == "HIGH").astype(float)
        if trials_name == "<50%":
            d = d[d.base < 0]
        d = d.rename(columns={"peak_2.0_2.8": "peak"})[["peak", "base", "HIGH", "pupil12", "subject", "trial"]].dropna()
        full = fit_fig3(d, "peak ~ base + HIGH * pupil12"); null = fit_fig3(d, "peak ~ base")
        out[f"fig3 {trials_name} trials, p base; evoked; interaction; LRT"] = (
            f"{full.pvalues['base']:.2g}; {full.pvalues['pupil12']:.2g}; {full.pvalues['HIGH:pupil12']:.2g}; "
            f"{stats.chi2.sf(2 * (full.llf - null.llf), 3):.2g}")
    # Figure 4
    dr = trx[trx.kind == "drc"]
    w4 = dr[dr.subject.isin(GC) & dr.treat.isin(["CTRL", "ZX1"]) & (dr.base_pct_max < 50)]
    r, fits, _ = gca(w4)
    out["fig4c terms p < 0.05 (written ref, factorial ref, written sum, factorial sum); c x ZX1 p"] = (
        f"{(r[terms] < 0.05).sum(1).tolist()}; {r.c_x_p.max():.2g}")
    main_terms = ["contrast_int_p", "contrast_slope_p", "x_int_p", "x_slope_p"]
    full_ = (r[terms] < 0.05).all(1)
    part_ = (r.c_x_p < 0.05) & ((r[main_terms] < 0.05).sum(1) >= 2)
    out["fig4c readings meeting the full criterion; the partial one (or the full)"] = (
        f"{int(full_.sum())} of 4; {int((part_ | full_).sum())} of 4")
    ph = posthoc_summary({k: fits[k] for k in [("written+cxt", "reference")]}).iloc[0]
    out["fig4 post hoc p: HIGH-LOW CTRL; HIGH-LOW ZX1; ZX1 on HIGH; ZX1 on LOW"] = (
        f"{ph['p']:.2g}; {ph['p ']:.2g}; {ph['p  ']:.2g}; {ph['p   ']:.2g}")
    k4 = dr[dr.subject.isin(GD) & dr.treat.isin(["CTRL", "ZX1"]) & (dr.base_pct_max < 50)]
    rk, _, _ = gca(k4)
    out["fig4f readings with all four absence terms p >= 0.05"] = f"{int((rk[absent] >= 0.05).all(1).sum())} of 4"
    out["fig4f KO ZX1 slope p (range); readings meeting the criterion"] = (
        f"{rk.x_slope_p.min():.2g} to {rk.x_slope_p.max():.2g}; "
        f"{int(((rk.x_slope_b < 0) & (rk.x_slope_p < 0.05) & (rk.x_slope_b.abs() < 0.5 * r.x_slope_b.abs())).sum())} of 4")
    # Figure 6
    st = trx[trx.kind.isin(["pt", "drc"])]
    def ci(mice):
        d = st[st.subject.isin(mice)]
        q = stats.linregress(d.rec_min, d.base_pct_max)
        tc = stats.t.ppf(0.975, len(d) - 2)
        return q.slope - tc * q.stderr, q.slope + tc * q.stderr, q.pvalue
    ac_ = ci(GROUPS["F"])
    zxs = {"a": ci(GC + GD), "b": ci(GC), "c": ci(sorted(set(GROUPS["B"]) | set(GC) | set(GD))),
           "d": ci(GROUPS["ZX1_drift"])}
    out["fig6a ACSF CI; ZX1 readings (a)-(d) CI, overlap with ACSF"] = f"({ac_[0]:.3f}, {ac_[1]:.3f}); " + "; ".join(
        f"({k}) ({z[0]:.3f}, {z[1]:.3f}) {max(z[0], ac_[0]) <= min(z[1], ac_[1])}" for k, z in zxs.items())
    a6 = dr[dr.subject.isin(GROUPS["F"]) & (dr.treat == "ACSF") & (dr.base_pct_max < 50)]
    ra_, _, _ = gca(a6, forms={"post only": "pupil ~ (tPoly1 + tPoly2) * c"}, codings=("reference",))
    out["fig6b (a) contrast intercept p; slope p"] = f"{ra_.contrast_int_p[0]:.2g}; {ra_.contrast_slope_p[0]:.2g}"
    rows_ = []
    for g, kinds_ in (("A", ["pt"]), ("B", ["pt"]), ("C", ["drc"]), ("D", ["drc"]), ("F", ["drc"])):
        for trt in ("CTRL", "all"):
            d = st[st.subject.isin(GROUPS[g]) & st.kind.isin(kinds_)]
            d = d[d.treat == "CTRL"] if trt == "CTRL" else d
            d50 = d[d.base_pct_max < 50]
            rows_.append({"r": f"{g} {trt}", "listed": g != "B" or trt == "CTRL", "n": len(d), "n50": len(d50),
                          "r_all": stats.pearsonr(d.base_pct_max, d.ev_1_2).statistic,
                          "p50": stats.pearsonr(d50.base_pct_max, d50.ev_1_2).pvalue})
    rr = pd.DataFrame(rows_)
    m6c = rr[(rr.n50 - 240).abs() <= 24]
    out["fig6c readings within 240 (n, p below 50%); listed readings significant"] = (
        "; ".join(f"{x.r}{'' if x.listed else ' (added)'} ({x.n50}, {x.p50:.2g})" for x in m6c.itertuples()) or "none") + (
        f"; {int((rr[rr.listed].p50 < 0.05).sum())} of {int(rr.listed.sum())}")
    mm = rr[((rr.n - 626).abs() <= 62.6) & ((rr.n50 - 384).abs() <= 38.4)]
    out["methods corr readings matching both counts (r all, p below 50%)"] = (
        "; ".join(f"{x.r} ({x.r_all:.3f}, {x.p50:.2g})" for x in mm.itertuples()) or "none")
    g6 = dr[dr.subject.isin(GC + GD) & (dr.treat == "CTRL") & (dr.contrast == "HIGH") & (dr.base_pct_max < 50)].copy()
    r6, _, _ = gca(g6, factor="genotype", level="ZnT3KO", forms={"genotype": "pupil ~ (tPoly1 + tPoly2) * x"}, codings=("reference",))
    out["fig6d genotype x tPoly1 b; p"] = f"{r6.x_slope_b[0]:.2f}; {r6.x_slope_p[0]:.2g}"
    pw = peak_rise(w4[(w4.treat == "ZX1") & (w4.contrast == "HIGH")])[0]
    pk = peak_rise(k4[(k4.treat == "CTRL") & (k4.contrast == "HIGH")])[0]
    pwc = peak_rise(w4[(w4.treat == "CTRL") & (w4.contrast == "HIGH")])[0]
    out["discussion peak rise WT ZX1 HIGH; KO CTRL HIGH; WT CTRL HIGH"] = f"{pw:.2f}; {pk:.2f}; {pwc:.2f}"
    return out

no_art = ["AA0319", "AA0334"]
VARIANTS = {
    "main": (tr, None),
    "round-1 cleaning": (prev, None),
    "single pass": (single, None),
    "round-1, without AA0319 and AA0334": (prev[~prev.subject.isin(no_art)], None),
    "main, group A without AA0335": (tr, [m for m in GROUPS["A"] if m != "AA0335"]),
    "steps counted after the spike filter": (narrow, None),
}
KEY = pd.DataFrame({name: key_results(trx, am) for name, (trx, am) in VARIANTS.items()})
with pd.option_context("display.max_colwidth", 200, "display.width", 400):
    for name in KEY.columns:
        print(f"== {name} ==")
        print(KEY[name].to_string())
        print()
print("trials below 50% of animal max: main", int((tr.base_pct_max < 50).sum()), "of", len(tr), "; round-1",
      int((prev.base_pct_max < 50).sum()), "of", len(prev), "; single pass", int((single.base_pct_max < 50).sum()), "of", len(single))


## Summary

| Claim | Outcome |
|---|---|
| fig1f-tone-response-inverse-u | reproduced |
| fig1g-reliability-intermediate | reproduced |
| fig1h-cgc-largest-intermediate | reproduced |
| fig1h-cgc-evident-all-bins | not reproduced |
| fig1i-baseline-contrast-interaction | reproduced |
| fig1i-contrast-effect-intermediate-only | not reproduced |
| fig2b-evoked-pd-whole-drc | not reproduced |
| fig2c-evoked-pd-1-2s | blocked: methods underspecified |
| fig2e-drc-responses-no-difference | not reproduced |
| fig3b-lme-coefficients | reproduced |
| fig3c-lme-likelihood-ratio | not reproduced |
| fig4c-wt-contrast-zx1-effects | partially reproduced |
| fig4-wt-posthoc | not reproduced |
| fig4f-ko-no-contrast-effect | blocked: methods underspecified |
| fig4f-ko-zx1-slope | not reproduced |
| fig5-contralateral-zx1-increases | blocked: data missing |
| fig5-ipsilateral-zx1-reduces | blocked: data missing |
| fig5c-no-dye-in-other-hemisphere | blocked: data missing |
| fig6a-baseline-pd-drift | partially reproduced |
| fig6b-acsf-contrast-effect | partially reproduced |
| fig6c-no-baseline-evoked-correlation | blocked: methods underspecified |
| methods-baseline-evoked-correlation | blocked: methods underspecified |
| fig6d-ko-reduced-high-contrast | reproduced |
| discussion-zx1-wt-like-ko | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

How the analysis differs from the paper's:

- The files hold the pupil radius per video frame, with dropped frames and tracking failures (up to 1.5e6
  pixels), not the DeepLabCut points. The Methods' single derivative pass leaves these failures in, so a
  spike filter was added, the pass repeated, trials with more than half their frames missing dropped, and
  trials whose raw trace steps by more than a factor 2 between tracked frames dropped (118 of 2113). This
  sets each animal's maximum PD, hence the baseline in % of maximum and the 50% selection. The step rule
  reads the raw trace and so drops trials in 15 mice, not only the two whose failure runs it targets;
  counted after the spike filter it drops 39 trials in 6 mice and gives the same animal maxima. Several
  outcomes depend on the cleaning (variants at the end): with the round-1 cleaning, which kept the step
  trials, fig4c-wt-contrast-zx1-effects would be not reproduced (no interaction); fig4-wt-posthoc fails
  under every cleaning but the narrower step rule, which meets it; with the single pass,
  fig6c-no-baseline-evoked-correlation would be not reproduced, fig6d-ko-reduced-high-contrast not
  reproduced (p = 0.20), and the fig1i interaction not significant (p = 0.17), which would make
  fig1i-baseline-contrast-interaction partially reproduced.
- The DRC trials without the tone switch contrast after 10 s (both orders, about equally often); a trial's
  contrast is that of its first 10 s, which covers the analysed 0 to 2 s. The 32 ZX1 DRC trials of AA0327
  start the sound 10 s after the first frame, and the pupil dilates after that onset.
- Groups are taken from the files' analysis labels; group A holds ten mice (paper eight; one is ZnT3-WT),
  group B twelve (paper fourteen C57BL6/J; six here are C57BL6/J), and the ZnT3-WT group five F and five M
  (paper six F and four M). Keeping the ZnT3-WT mouse AA0335 in group A is an assumption the Fig. 2b and
  Fig. 3b outcomes rest on: without it group A's Fig. 2b trial test becomes significant (p = 0.028), so the
  readings would disagree, and the Fig. 3 baseline coefficient is not significant in the trials below 50%
  (p = 0.087), which would make fig3b-lme-coefficients partially reproduced.
- LME tests are Wald z tests from statsmodels, not Satterthwaite F tests; the Fig. 3 random intercepts
  for baseline and evoked PD are fitted as one trial intercept.
- Recording time for Fig. 6a comes from the .tif creation times, since the table's starting times wrap
  by one day for trials recorded before the reference file.

What was found about the dataset: the files carry labels naming the paper's analyses, a per-trial table
of sounds and treatments, ROI fluorescence and pupil radius, enough to test 21 of 24 claims. The
widefield experiments of Figure 5 are not in it. Pupil traces hold tracking failures, in two mice as runs
at a second level; 25 pupil series have a length that does not match their trial; every pupil series's
description names the wrong TwoPhotonSeries; the stimulus table's starting times wrap by 24 h for 84
trials; the contrast-switching DRC stimulus is not described in the paper; and group sizes differ from
the paper.


In [ ]:
print(f"total run time {(time.time() - T0) / 60:.1f} min")